# Generating the catalogue caches

This section discusses the catalog fields that are taken from each simulation and how these fields are used in the pipeline. To ensure a fair comparison between the simulations, we recompute the stellar properties using comparable definitions.

This optional notebook reuses the existing FLAMINGO and TNG300 selection functions. It writes the local sample/pair pickle files, aligned TNG arrays, particle-feature caches and bound-mass sidecars. These files are not included in Git. The main notebook sequence uses the saved result tables and does not require this reconstruction or a Roihu job.

Read [the data guide](../docs/data.ipynb) before enabling any stage. Raw FLAMINGO selection reads a large parent catalogue; TNG particle cutouts require a personal API key. All stages are off by default. Use a fresh local cache folder and retain the original inputs.


In [1]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
ROOT = next((p for p in candidates if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook inside the repository with pyproject.toml.')
PROJECT = ROOT / 'research'
os.environ['LG_PROJECT'] = str(PROJECT)

REBUILD = False
# Add only the stages you intend to run, then set REBUILD = True.
# Valid stages: flamingo, tng, particles, metallicity, manifest, sidecars, validate.
STAGES = set()
SAVE_LARGE_FLAMINGO_PARENT = False

allowed = {'flamingo', 'tng', 'particles', 'metallicity', 'manifest', 'sidecars', 'validate'}
if not STAGES <= allowed:
    raise ValueError(f'Unknown stages: {STAGES - allowed}')
print('Local cache root:', PROJECT.relative_to(ROOT))
print('Enabled stages:', sorted(STAGES) if REBUILD else 'none (read-only default)')


Local cache root: research
Enabled stages: none (read-only default)


## Prepare the original helper paths

The source is under `research/analogues/` and `research/TNG300/analogues/`. Generated caches sit beside these helpers so that the original particle scripts keep their expected relative paths. `TNG_BASE` should point to the local TNG300-1 `outputs` folder containing `groups_099`. Set it in your shell before starting Jupyter. Keep the API key out of notebook source and saved outputs.

The small functions below only handle file output. They do not change the selection, feature definitions or physical calculations. Existing arrays are checked and retained. An existing pickle output stops a fresh catalogue build before loading the large source catalogue.


In [2]:
import json
import pickle
import hashlib

def save_array(path, values):
    import numpy as np
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        np.testing.assert_array_equal(np.load(path), values)
        print('Verified existing array:', path.relative_to(PROJECT))
        return
    with path.open('xb') as stream:
        np.save(stream, values)

def save_pickle_new(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('xb') as stream:
        pickle.dump(value, stream, protocol=pickle.HIGHEST_PROTOCOL)

def save_json_checked(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        if json.loads(path.read_text(encoding='utf-8')) != value:
            raise FileExistsError(f'Preserve the existing metadata and choose a new cache version: {path}')
        return
    with path.open('x', encoding='utf-8') as stream:
        json.dump(value, stream, indent=2)

if REBUILD:
    sys.path.insert(0, str(PROJECT))
    sys.path.insert(0, str(PROJECT / 'analogues'))


## FLAMINGO L1_m8, z=0

The SOAP catalogue is `FLAMINGO/L1_m8/L1_m8/SOAP-HBT/halo_properties_0078.hdf5`. The pipeline uses `hdfstream.open('cosma', '/')` through its existing helper. FLAMINGO mass values are multiplied by `1e10` to obtain solar masses, without dividing by h. Positions are converted from Mpc to kpc.

The original selected sample has 3,366,610 galaxies and 142,713 final pairs. Counts alone do not establish exact input identity. The full parent `fields` dictionary was about 8.63 GB as a saved pickle, and is only saved here if `SAVE_LARGE_FLAMINGO_PARENT=True`.


In [3]:
if REBUILD and 'flamingo' in STAGES:
    from flamingo_lg_pipeline import (
        open_default_soap, load_selection_config,
        run_flamingo_pipeline_from_config,
    )
    cache = PROJECT / 'analogues/cache/run_001_mstar2e10'
    destinations = [cache / 'flamingo_sample.pkl', cache / 'flamingo_pairs.pkl']
    if any(p.exists() for p in destinations):
        raise FileExistsError('Preserve the existing FLAMINGO pickles; use their cached analysis or choose a new cache version.')
    cfg = load_selection_config(str(PROJECT / 'analogues/flamingo_selection_config.json'))
    fields, sample, pairs = run_flamingo_pipeline_from_config(
        config=cfg, soap_file=open_default_soap(), verbose=True,
    )
    save_pickle_new(destinations[0], sample)
    save_pickle_new(destinations[1], pairs)
    save_json_checked(cache / 'config.json', cfg)
    if SAVE_LARGE_FLAMINGO_PARENT:
        save_pickle_new(cache / 'flamingo_fields.pkl', fields)
    print('Sample galaxies:', len(sample['row_index']))
    print('Final pairs:', len(pairs['i']))
else:
    print('FLAMINGO catalogue reconstruction is disabled.')


FLAMINGO catalogue reconstruction is disabled.


## TNG300-1, snapshot 99

Download the snapshot 99 FoF/Subfind group catalogues from the [official TNG300-1 page](https://www.tng-project.org/data/downloads/TNG300-1/) and install the [Illustris Python helpers](https://www.tng-project.org/data/docs/scripts/). The original corrected sample has 45,392 galaxies and 1,530 pairs, with 1,253 same-host and 277 different-host pairs.

The old `rebuild_comparison.py` is a migration that requires older cache files. A fresh reconstruction instead calls its existing `AnalogueSample` class directly. The export below retains the original checked correction for the historical internal `log_tot_virial_mass` conversion. `pairs_i` and `pairs_j` are sample indices; `keep_idx[pairs_i]` and `keep_idx[pairs_j]` are global subhalo IDs.


In [4]:
if REBUILD and 'tng' in STAGES:
    import numpy as np
    import illustris_python as il
    from dataclasses import asdict
    from TNG300.analogues import AnalogueSample, SelectionConfig, FilterConfig

    if not os.environ.get('TNG_BASE'):
        raise RuntimeError('Set TNG_BASE to the TNG300-1 outputs folder containing groups_099.')
    tng_base = str(Path(os.environ['TNG_BASE']).resolve())
    tng_sample = AnalogueSample(
        base_path=tng_base, snap=99, verbose=True,
        selection_config=SelectionConfig(2e10, 5e11, 500., 1000., .68),
        filter_config=FilterConfig(
            vr_min=-400, vr_max=0, vt_min=0, vt_max=300,
            v_tot_min=0, v_tot_max=500,
            third_massive_factor=1.5, density_radius=2000,
            intruder_factor=.5,
        ),
    )
    s, p = tng_sample._sample, tng_sample.pairs
    header = il.groupcat.loadHeader(tng_base, 99)
    h = float(header['HubbleParam'])
    log_m200c = p.log_tot_virial_mass - 10 - np.log10(1 / h)
    physical_mass = np.where(p.have_same_host, s.m200c[p.i], s.m200c[p.i] + s.m200c[p.j])
    np.testing.assert_allclose(log_m200c, np.log10(physical_mass), atol=3e-6)
    native_sigma = il.groupcat.loadSubhalos(tng_base, 99, fields='SubhaloVelDisp')
    arrays = {
        'keep_idx': s.keep_idx, 'stellar_mass': s.m_stellar,
        'grnr': s.grnr, 'g_r': s.g_band - s.r_band,
        'pairs_i': p.i, 'pairs_j': p.j,
        'separation': p.separation, 'vr': p.vr, 'vt': p.vt,
        'log_m200c': log_m200c, 'log_mass_ratio': p.log_mass_ratio,
        'have_same_host': p.have_same_host,
        'is_blue_blue': p.is_blue_blue, 'is_red_red': p.is_red_red,
        'vel_disp': native_sigma[s.keep_idx],
    }
    tng_dir = PROJECT / 'analogues/tng_for_comparison_v3'
    for name, values in arrays.items():
        save_array(tng_dir / f'{name}.npy', values)
    same = p.have_same_host.astype(bool)
    volume = (float(header['BoxSize']) / h / 1000.) ** 3
    selection_build = {
        'run_version': 'reconstructed-from-public-catalogues',
        'snapshot': 99, 'h': h,
        'BoxSize_ckpc_h': float(header['BoxSize']),
        'volume_mpc3': volume,
        'selection': asdict(tng_sample._selection_config),
        'filters': asdict(tng_sample._filter_config),
        'intruder_mass': 'total bound SubhaloMass', 'vr': 'peculiar km/s',
        'sample_size': len(s.keep_idx), 'pair_count': len(p.i),
        'same_host': int(same.sum()), 'different_host': int((~same).sum()),
        'pair_density_mpc3': len(p.i) / volume,
    }
    save_json_checked(tng_dir / 'selection_build.json', selection_build)
    print({k: selection_build[k] for k in ('sample_size', 'pair_count', 'same_host', 'different_host')})
else:
    print('TNG catalogue reconstruction is disabled.')


TNG catalogue reconstruction is disabled.


## Matched stellar velocity dispersion, kappa and metallicity

To ensure a fair comparison between the simulations, we recompute stellar velocity dispersion, co-rotation and metallicity using the original particle-feature scripts. They use all bound stellar particles, without an aperture cut. They exclude TNG wind particles with `GFM_StellarFormationTime <= 0` and require at least 100 valid stars.

The scripts request only final-pair members, checkpoint their lookups, and can take time. A cached `NaN` counts as a cached entry in the existing resume logic. The validation step below reports missing values; preserve the old cache when developing a retry strategy. `TNG_BASE` is also required for the kappa script.

If the environment key is absent, the opt-in cell prompts privately. The value is held only in process memory and is never displayed or written to notebook source.


In [5]:
if REBUILD and 'particles' in STAGES:
    import subprocess
    import getpass
    if not os.environ.get('TNG_BASE'):
        raise RuntimeError('Set TNG_BASE before requesting the TNG stellar features.')
    if not os.environ.get('TNG_API_KEY'):
        os.environ['TNG_API_KEY'] = getpass.getpass('Personal TNG API key: ')
    if not os.environ['TNG_API_KEY']:
        raise RuntimeError('A personal TNG_API_KEY is required.')
    for name in ('compute_sigma_star.py', 'compute_kappa_corot.py', 'compute_metallicity.py'):
        subprocess.run([sys.executable, '-B', str(PROJECT / 'analogues' / name)],
                       cwd=PROJECT, env=os.environ.copy(), check=True)
else:
    print('TNG particle API requests are disabled.')


TNG particle API requests are disabled.


## Shared metallicity representation

Both simulations use `log10(Z / 0.0127)`. The per-simulation z-scores below are descriptive comparison quantities. NPE uses the dex feature and a scaler fit on training rows; the descriptive full-population scalers are not used as a replacement.


In [6]:
if REBUILD and 'metallicity' in STAGES:
    import numpy as np
    cache = PROJECT / 'analogues/cache/run_001_mstar2e10'
    target = PROJECT / 'analogues/cache/tng_features_v3/metallicity_log_zscore.pkl'
    if target.exists():
        raise FileExistsError('The shared metallicity cache already exists; preserve it and use validation.')
    with (cache / 'flamingo_sample.pkl').open('rb') as stream:
        fs = pickle.load(stream)
    with (cache / 'flamingo_pairs.pkl').open('rb') as stream:
        fp = pickle.load(stream)
    tng_dir = PROJECT / 'analogues/tng_for_comparison_v3'
    tm = np.load(tng_dir / 'metallicity.npy')
    ti, tj = np.load(tng_dir / 'pairs_i.npy'), np.load(tng_dir / 'pairs_j.npy')
    Z_sun = .0127
    with np.errstate(divide='ignore', invalid='ignore'):
        fdex = np.where(fs['metallicity'] > 0, np.log10(fs['metallicity'] / Z_sun), np.nan)
        tdex = np.where(tm > 0, np.log10(tm / Z_sun), np.nan)
    fpop, tpop = fdex[np.r_[fp['i'], fp['j']]], tdex[np.r_[ti, tj]]
    fpop, tpop = fpop[np.isfinite(fpop)], tpop[np.isfinite(tpop)]
    mu_f, sd_f = fpop.mean(), fpop.std(ddof=1)
    mu_t, sd_t = tpop.mean(), tpop.std(ddof=1)
    met = {
        'Z_sun': Z_sun,
        'mu_flam': mu_f, 'std_flam': sd_f, 'mu_tng': mu_t, 'std_tng': sd_t,
        'flamingo_met_dex': fdex, 'tng_met_dex': tdex,
        'flamingo_met_scaled': (fdex - mu_f) / sd_f,
        'tng_met_scaled': (tdex - mu_t) / sd_t,
    }
    save_pickle_new(target, met)
    print('Saved shared metallicity cache.')
else:
    print('Shared metallicity-cache generation is disabled.')


Shared metallicity-cache generation is disabled.


## Record the TNG selection and validate feature alignment

The manifest records this reconstruction as a new input version. Its hash includes every `.npy` file in filename order. A new file hash may differ because of feature availability away from the selected pairs or serialization details. Do not rename that new hash to the old fixed value.

The final-pair member checks require finite matched stellar properties. Pair counts and finite values are necessary checks, but exact scientific reproduction also requires the fixed pair-frame identity and split membership.


In [7]:
if REBUILD and 'manifest' in STAGES:
    import numpy as np
    from comparison_utils import validate_selection
    tng_dir = PROJECT / 'analogues/tng_for_comparison_v3'
    manifest = json.loads((tng_dir / 'selection_build.json').read_text(encoding='utf-8'))
    digest = hashlib.sha256()
    for path in sorted(tng_dir.glob('*.npy')):
        digest.update(path.name.encode())
        digest.update(path.read_bytes())
    manifest['dataset_sha256'] = digest.hexdigest()
    data = {path.stem: np.load(path) for path in tng_dir.glob('*.npy')}
    validate_selection(data, manifest)
    member_index = np.unique(np.r_[data['pairs_i'], data['pairs_j']])
    global_ids = data['keep_idx'][member_index]
    missing = {}
    for name in ('tng_sigma_star.pkl', 'tng_kappa_corot.pkl'):
        with (PROJECT / 'analogues/cache/tng_features_v3' / name).open('rb') as stream:
            lookup = pickle.load(stream)
        missing[name] = [int(i) for i in global_ids if not np.isfinite(lookup.get(int(i), np.nan))]
    missing['metallicity.npy'] = global_ids[~np.isfinite(data['metallicity'][member_index])].astype(int).tolist()
    manifest['missing_custom_features'] = missing
    print('Missing final-pair member values:', {k: len(v) for k, v in missing.items()})
    if any(missing.values()):
        raise ValueError('Resolve nonfinite matched features before treating this as a complete-case sample.')
    save_json_checked(tng_dir / 'selection_manifest.json', manifest)
    print('Saved aligned reconstruction manifest:', manifest['dataset_sha256'])
else:
    print('Manifest generation is disabled.')


Manifest generation is disabled.


## Bound-subhalo mass sidecars

The alternative target is the sum of the total bound subhalo masses. It is separate from M200c. The existing exporter writes each endpoint mass and its sum, records the source fields and unit conversions, and refuses incompatible overwrites.


In [8]:
if REBUILD and 'sidecars' in STAGES:
    import subprocess
    if not os.environ.get('TNG_BASE'):
        raise RuntimeError('Set TNG_BASE before exporting the TNG bound-mass sidecar.')
    subprocess.run([
        sys.executable, '-B', str(PROJECT / 'scripts/export_bound_subhalo_masses.py'),
        '--project', str(PROJECT), '--tng-base', os.environ['TNG_BASE'],
        '--simulation', 'both',
    ], cwd=ROOT, env=os.environ.copy(), check=True)
else:
    print('Bound-mass sidecar generation is disabled.')


Bound-mass sidecar generation is disabled.


## Validate the common pair tables

The existing `lg_npe.data` loaders provide one row per pair. Same-host pairs use a single host's M200c, whilst different-host pairs sum the independent M200c values. Both pair radial velocities are peculiar velocities.

The exact original final identities cover `pair_index, id_a, id_b, host_i, host_j, same_host, separation, vr, vt, mstar_a, mstar_b, log_m200c`. If a fresh input does not match the fixed identity, retain it as a new version and investigate; do not round values, change split membership or replace the original results.


In [9]:
if REBUILD and 'validate' in STAGES:
    for location in (ROOT / 'src', PROJECT / 'src'):
        if (location / 'lg_npe').is_dir():
            sys.path.insert(0, str(location))
    from lg_npe.final_design import load_final_frames, frame_digest
    frames = load_final_frames(PROJECT)
    expected = {
        'flamingo': 'a23f726a1869af695bde0f1c62527ec595f5e1a59522e0f3b55b517b8811e69a',
        'tng': '6394a47d6f81aed87f0fdc0235ddd2d94bd1f1fc35a9669f93803d85bf75ffb2',
    }
    matching = {}
    for domain, f in frames.items():
        print(domain, 'pairs:', len(f), 'same-host:', int(f.same_host.sum()),
              'different-host:', int((~f.same_host).sum()))
        actual = frame_digest(f)
        matching[domain] = actual == expected[domain]
        print('Final input identity:', actual, 'matches frozen:', matching[domain])
    if not all(matching.values()):
        raise ValueError('Reconstructed inputs differ from frozen final identities. Preserve this as a new version and investigate before using frozen splits or checkpoints.')
    print('Both complete-case pair frames match the frozen final identities.')
else:
    print('Pair-table validation is disabled.')


Pair-table validation is disabled.


## Continue with the thesis analysis

The catalogue caches do not contain trained posteriors or bootstrap outcomes. The main notebooks read the included result tables. New calculations using the exact saved posterior draws require the optional archived artifact bundle; new NPE fits require the training workflow and its scientific environment.

The completed thesis results and the later classifier-mixture diagnostic already exist. A Roihu rerun is unnecessary for documenting, reading or replotting them. See [the data guide](../docs/data.ipynb) for the expected local files and [the README](../README.md) for the main reading order.
